# 呉市場 生鮮水産物 PDF → CSV 変換
PDFをアップロードするとCSVに変換します。

In [ ]:
!pip install pdfplumber -q

In [ ]:
from google.colab import files
uploaded = files.upload()  # PDFをアップロード
PDF_PATH = list(uploaded.keys())[0]
OUTPUT_PATH = PDF_PATH.replace('.pdf', '.csv').replace('.PDF', '.csv')
print(f'アップロード: {PDF_PATH}')
print(f'出力先: {OUTPUT_PATH}')

In [ ]:
import re
import csv
from pathlib import Path
import pdfplumber

REIWA_BASE = 2018

NAME_MAP = {
    'まなかつ': 'まなかつお',
    'その他淡': 'その他淡水魚類',
    'その他貝': 'その他貝類',
    'その他鮮': 'その他鮮魚類',
}

ORIGINS = {'合計', '広島', '愛媛', '東京', '福岡', '山口', '三重', '大分', '熊本', '島根', '福井'}

SKIP_TEXTS = {'生鮮水産物', '品目産地別・月別取扱高表', '単位', '上段', '下段', '数量', '金額',
              '令和', '月別', '品目', '産地別', '合', '計'}

DATA_X_OFFSET = 20.3

def detect_year(pdf_path):
    m = re.search(r'令和([０-９\d]+)年', pdf_path)
    if m:
        num = m.group(1).translate(str.maketrans('０１２３４５６７８９', '0123456789'))
        return REIWA_BASE + int(num)
    with pdfplumber.open(pdf_path) as pdf:
        for w in pdf.pages[0].extract_words()[:30]:
            m = re.match(r'令和([０-９\d]+)年', w['text'])
            if m:
                num = m.group(1).translate(str.maketrans('０１２３４５６７８９', '0123456789'))
                return REIWA_BASE + int(num)
    return 2025

def is_number(text):
    return bool(re.match(r'^[\d,]+$', text.strip()))

def parse_int(text):
    return int(text.replace(',', ''))

def get_month_x_ranges(words):
    MONTH_CHARS = {'１','２','３','４','５','６','７','８','９','１０','１１','１２'}
    MONTH_ORDER = ['１','２','３','４','５','６','７','８','９','１０','１１','１２']
    header_top = None
    month_positions = {}
    for w in words:
        if w['text'] in MONTH_CHARS:
            if header_top is None:
                header_top = w['top']
            if abs(w['top'] - header_top) < 5:
                month_positions[w['text']] = w['x0']
    if len(month_positions) < 12:
        return []
    centers = [month_positions[m] + DATA_X_OFFSET for m in MONTH_ORDER if m in month_positions]
    if len(centers) < 12:
        return []
    ranges = []
    for i, center in enumerate(centers):
        x_start = (centers[i-1] + center) / 2 if i > 0 else center - 30
        x_end = (center + centers[i+1]) / 2 if i < len(centers)-1 else center + 30
        ranges.append((x_start, x_end))
    return ranges

def assign_to_month(x0, month_ranges):
    for i, (x_start, x_end) in enumerate(month_ranges):
        if x_start <= x0 < x_end:
            return i + 1
    return None

def group_rows_by_top(words, tolerance=2.0):
    if not words:
        return []
    rows = []
    current_row = [words[0]]
    for w in words[1:]:
        if abs(w['top'] - current_row[0]['top']) <= tolerance:
            current_row.append(w)
        else:
            rows.append(sorted(current_row, key=lambda x: x['x0']))
            current_row = [w]
    rows.append(sorted(current_row, key=lambda x: x['x0']))
    return rows

def parse_page(page, month_ranges, year):
    records = []
    words = page.extract_words()
    rows = group_rows_by_top(words)
    current_item = None
    current_origin = None
    pending_row = None
    for row in rows:
        texts = [w['text'] for w in row]
        row_text = ''.join(texts)
        if any(t in SKIP_TEXTS for t in texts) and not any(is_number(t) for t in texts):
            continue
        if '令和' in row_text or '№' in row_text:
            continue
        new_item = None
        new_origin = None
        for i, t in enumerate(texts):
            if t in ORIGINS:
                new_origin = t
                if i > 0 and texts[i-1] not in ORIGINS and not is_number(texts[i-1]):
                    candidate = re.sub(r'\s+', '', texts[i-1])
                    candidate = NAME_MAP.get(candidate, candidate)
                    if candidate:
                        new_item = candidate
                break
        if new_item:
            current_item = new_item
        if new_origin:
            current_origin = new_origin
        num_words = [w for w in row if is_number(w['text'])]
        if not num_words:
            continue
        if pending_row is not None:
            kin_words = num_words
            qty_words = pending_row
            qty_by_month = {}
            kin_by_month = {}
            for w in qty_words:
                m = assign_to_month(w['x0'], month_ranges)
                if m:
                    qty_by_month[m] = parse_int(w['text'])
            for w in kin_words:
                m = assign_to_month(w['x0'], month_ranges)
                if m:
                    kin_by_month[m] = parse_int(w['text'])
            if current_item and current_origin:
                for month in range(1, 13):
                    qty = qty_by_month.get(month)
                    kin = kin_by_month.get(month)
                    if qty is not None or kin is not None:
                        records.append({
                            '品目': current_item, '産地': current_origin,
                            '年': year, '月': month,
                            '数量_kg': qty, '金額_円': kin,
                        })
            pending_row = None
        elif current_item and current_origin and num_words:
            pending_row = num_words
    return records

# --- 実行 ---
year = detect_year(PDF_PATH)
print(f'検出年度: {year}年')

all_records = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        words = page.extract_words()
        month_ranges = get_month_x_ranges(words)
        if not month_ranges:
            print(f'  ページ{page_num}: ヘッダー未検出 スキップ')
            continue
        records = parse_page(page, month_ranges, year)
        all_records.extend(records)
        print(f'  ページ{page_num}: {len(records)}件')

print(f'\n総レコード数: {len(all_records)}')
print(f'品目数: {len(set(r["品目"] for r in all_records))}')

with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['品目','産地','年','月','数量_kg','金額_円'])
    writer.writeheader()
    writer.writerows(all_records)

print(f'CSV保存完了: {OUTPUT_PATH}')

In [ ]:
# CSVをダウンロード
files.download(OUTPUT_PATH)